# Chapter 6: Algorithm Chains and Pipelines
## *Introduction to Machine Learning with Python*  Müller & Guido

---
> **Ringkasan Bab:** Bab ini memperkenalkan konsep **Pipeline**  cara menggabungkan preprocessing dan model dalam satu objek yang bersih dan aman. Pipeline mencegah *data leakage*, menyederhanakan kode, dan memungkinkan hyperparameter tuning yang benar.

## 6.1 Masalah Tanpa Pipeline: Data Leakage

**Data leakage** terjadi ketika informasi dari test set "bocor" ke proses training, membuat evaluasi menjadi optimistis secara tidak adil.

###  Cara SALAH  Data Leakage:

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=0
)

#  SALAH: fit scaler pada SEMUA data (train + test) sebelum split
scaler = StandardScaler()
scaler.fit(cancer.data)                    # <-- LEAKAGE: test set ikut di-fit!
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

svm = SVC().fit(X_train_scaled, y_train)
print(f" SALAH (data leakage) - Test Accuracy: {svm.score(X_test_scaled, y_test):.3f}")

In [ ]:
#  BENAR: fit scaler HANYA pada training data
scaler_correct = StandardScaler()
scaler_correct.fit(X_train)                # <-- fit hanya pada training
X_train_scaled_c = scaler_correct.transform(X_train)
X_test_scaled_c  = scaler_correct.transform(X_test)  # transform saja, tidak fit

svm2 = SVC().fit(X_train_scaled_c, y_train)
print(f" BENAR (no leakage)  - Test Accuracy: {svm2.score(X_test_scaled_c, y_test):.3f}")

### 6.1.1 Data Leakage yang Lebih Halus (saat Cross-Validation)

Bahkan jika kita fit scaler hanya pada training data, jika kita melakukan preprocessing **di luar** cross-validation loop, tetap terjadi leakage!

> **Pipeline menyelesaikan masalah ini secara otomatis.** Ia memastikan setiap langkah preprocessing di-fit hanya pada data training di setiap fold.

In [ ]:
#  SALAH: scale dulu LALU cross-validate
scaler_bad = StandardScaler().fit(X_train)
X_train_scaled_bad = scaler_bad.transform(X_train)

# Saat CV, setiap fold validation set sudah "terkontaminasi" oleh fit scaler
cv_scores_bad = cross_val_score(SVC(), X_train_scaled_bad, y_train, cv=5)
print(f" SALAH CV (leakage): {cv_scores_bad.mean():.4f} ± {cv_scores_bad.std():.4f}")

## 6.2 Pipeline: Solusi Bersih dan Aman

**Pipeline** menggabungkan serangkaian langkah transformasi dan estimator akhir dalam satu objek.

**Cara kerja Pipeline saat `fit(X_train, y_train)`:**
1. Panggil `fit_transform` pada setiap step transformasi
2. Pasang output ke step berikutnya
3. Panggil `fit` pada estimator terakhir

**Cara kerja Pipeline saat `predict(X_test)`:**
1. Panggil `transform` (bukan `fit_transform`) pada setiap step
2. Pasang output ke step berikutnya
3. Panggil `predict` pada estimator terakhir

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Buat pipeline: Scaling → SVM
pipe = Pipeline([
    ('scaler', StandardScaler()),   # langkah 1: normalisasi
    ('svm',    SVC())               # langkah 2: model
])

# Sama seperti model biasa: fit, predict, score
pipe.fit(X_train, y_train)
print(f"✅ Pipeline Test Accuracy: {pipe.score(X_test, y_test):.3f}")

# Pipeline dengan cross-validation — AMAN!
cv_scores_pipe = cross_val_score(pipe, X_train, y_train, cv=5)
print(f"✅ Pipeline CV Score: {cv_scores_pipe.mean():.4f} ± {cv_scores_pipe.std():.4f}")

In [ ]:
# Cara alternatif: make_pipeline (nama step otomatis)
from sklearn.pipeline import make_pipeline

pipe2 = make_pipeline(StandardScaler(), SVC())
print("Pipeline steps:", pipe2.steps)
print("Nama steps    :", pipe2.named_steps.keys())

## 6.3 Pipeline dengan GridSearchCV

Pipeline + GridSearchCV = kombinasi yang sangat kuat dan benar untuk hyperparameter tuning.

**Notasi parameter dalam Pipeline:** `nama_step__nama_parameter`

In [ ]:
# Definisikan pipeline
pipe_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC())
])

# Parameter grid — gunakan format: nama_step__parameter
param_grid = {
    'svm__C'    : [0.001, 0.01, 0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1],
}

grid_search = GridSearchCV(pipe_cv, param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Parameter terbaik :", grid_search.best_params_)
print(f"CV Score terbaik  : {grid_search.best_score_:.3f}")
print(f"Test Score        : {grid_search.score(X_test, y_test):.3f}")

## 6.4 Pipeline dengan Multiple Preprocessing Steps

Pipeline bisa memiliki **banyak langkah** preprocessing sebelum model akhir.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Pipeline: Imputer → Scaler → PCA → Classifier
pipe_multi = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),    # isi missing values
    ('scaler',  StandardScaler()),                  # normalisasi
    ('pca',     PCA(n_components=10)),              # reduksi dimensi
    ('rf',      RandomForestClassifier(n_estimators=100, random_state=42)),  # model
])

# Cek pipeline
print("Langkah-langkah pipeline:")
for i, (name, step) in enumerate(pipe_multi.steps):
    print(f"  {i+1}. [{name}] → {type(step).__name__}")

# Fit dan evaluasi
pipe_multi.fit(X_train, y_train)
print(f"\nAkurasi Test: {pipe_multi.score(X_test, y_test):.3f}")

In [ ]:
# Grid search pada pipeline dengan banyak step
param_grid_multi = {
    'pca__n_components' : [5, 10, 15, 20],
    'rf__n_estimators'  : [50, 100, 200],
    'rf__max_depth'     : [None, 3, 5],
}

grid_multi = GridSearchCV(pipe_multi, param_grid_multi, cv=5, n_jobs=-1)
grid_multi.fit(X_train, y_train)

print("Parameter terbaik:", grid_multi.best_params_)
print(f"Test Score       : {grid_multi.score(X_test, y_test):.3f}")

## 6.5 ColumnTransformer: Preprocessing Berbeda per Kolom

Data dunia nyata sering punya campuran fitur numerik dan kategorik yang perlu preprocessing berbeda.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

# Dataset campuran
np.random.seed(42)
n = 300
df = pd.DataFrame({
    'usia'       : np.random.randint(20, 65, n),
    'pendapatan' : np.random.normal(5000, 2000, n).clip(500),
    'pengalaman' : np.random.randint(0, 30, n),
    'kota'       : np.random.choice(['Jakarta', 'Bandung', 'Surabaya'], n),
    'pendidikan' : np.random.choice(['SMA', 'S1', 'S2'], n),
    'target'     : np.random.randint(0, 2, n)
})

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Definisikan kolom
num_features = ['usia', 'pendapatan', 'pengalaman']
cat_features = ['kota', 'pendidikan']

# Preprocessing untuk tiap tipe kolom
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse=False))
])

# Gabungkan dalam ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features),
])

# Full pipeline
full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   GradientBoostingClassifier(random_state=42))
])

full_pipe.fit(X_train, y_train)
print(f"Pipeline Lengkap (Numerik + Kategorik)")
print(f"Akurasi Training: {full_pipe.score(X_train, y_train):.3f}")
print(f"Akurasi Test    : {full_pipe.score(X_test, y_test):.3f}")

##  Ringkasan Bab 6

| Konsep | Penjelasan |
|---|---|
| **Data Leakage** | Informasi test set bocor ke training → evaluasi optimistis |
| **Pipeline** | Rangkaian preprocessing + model dalam satu objek aman |
| **make_pipeline** | Cara cepat membuat pipeline, nama step otomatis |
| **Pipeline + CV** | Preprocessing dilakukan dengan benar di setiap fold |
| **Pipeline + GridSearch** | Notasi `step__param` untuk tuning |
| **ColumnTransformer** | Terapkan preprocessing berbeda ke kolom berbeda |

**Best Practice:**
```python
pipe = Pipeline([('scaler', StandardScaler()), ('model', SVC())])
GridSearchCV(pipe, {'model__C': [0.1, 1, 10]}, cv=5).fit(X_train, y_train)
```

---
 **Next:** Chapter 7  Working with Text Data